In [1]:
!pip install protobuf==3.20.3 --quiet
!pip install transformers datasets --upgrade --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, bu

In [2]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

csv_path = '/kaggle/input/brazilian-ecommerce/olist_order_reviews_dataset.csv'
df = pd.read_csv(csv_path)
df = df.dropna(subset=['review_comment_message'])
df['review_comment_title'] = df['review_comment_title'].fillna('')
df['full_text'] = df['review_comment_title'] + ". " + df['review_comment_message']
df = df[df['review_score'] != 3]
df['label'] = np.where(df['review_score'] >= 4, 1, 0)

print(f"Training Dataset Size: {len(df)}")

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
train_dataset = Dataset.from_pandas(train_df[['full_text', 'label']])
val_dataset = Dataset.from_pandas(val_df[['full_text', 'label']])

model_name = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["full_text"], padding="max_length", truncation=True, max_length=128)

print("Processing Tokenization...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch", 
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    report_to="none"
)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

print("Start full-scale training...")
trainer.train()

2025-12-03 12:50:29.906442: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764766230.088368      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764766230.139377      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Training Dataset Size: 37420


tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Start full-scale training...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.154700,0.145882,0.958578,0.970425,0.980150,0.960892
2,0.115200,0.146483,0.958445,0.970597,0.971423,0.969771


TrainOutput(global_step=3742, training_loss=0.14594519679906978, metrics={'train_runtime': 836.3641, 'train_samples_per_second': 71.586, 'train_steps_per_second': 4.474, 'total_flos': 3938246276628480.0, 'train_loss': 0.14594519679906978, 'epoch': 2.0})

In [3]:
save_path = "./Olist_bert_final_model"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model has been saved to: {save_path}")

from transformers import pipeline

classifier = pipeline("sentiment-analysis", model=save_path, tokenizer=save_path)

def predict_review(text):
    result = classifier(text)
    label = result[0]['label']
    score = result[0]['score']
    meaning = "Positive (Good Review)" if label == "LABEL_1" else "Negative (Bad Review)"
    print(f"Comment: {text}")
    print(f"Predict: {meaning} (Accuracy: {score:.4f})\n")

#Testing

predict_review("A embalagem é muito boa, a entrega foi eficiente, o vendedor é confiável e o envio foi muito rápido. Além disso, o preço é bem mais em conta do que nos concorrentes.") 
# (The packaging is great, delivery was efficient, and the seller is trustworthy with very fast shipping. Plus, the price is much more affordable than other stores.)

predict_review("Demorou para enviar e o produto que chegou é diferente do que eu comprei.") 
# (Late shipment, and the product I received is different from what I ordered.)

predict_review("O vendedor enviou rápido e o atendimento foi muito educado, mas o produto veio com defeito e não funciona.") 
# (The seller shipped quickly and customer service was very polite, but the product is defective and unusable.)


Model has been saved to: ./Olist_bert_final_model


Device set to use cuda:0


Comment: A embalagem é muito boa, a entrega foi eficiente, o vendedor é confiável e o envio foi muito rápido. Além disso, o preço é bem mais em conta do que nos concorrentes.
Predict: Positive (Good Review) (Accuracy: 0.9983)

Comment: Demorou para enviar e o produto que chegou é diferente do que eu comprei.
Predict: Negative (Bad Review) (Accuracy: 0.9902)

Comment: O vendedor enviou rápido e o atendimento foi muito educado, mas o produto veio com defeito e não funciona.
Predict: Negative (Bad Review) (Accuracy: 0.9520)



In [4]:
import pandas as pd
from transformers import pipeline
from tqdm.auto import tqdm
import torch

df_pb = pd.read_csv('/kaggle/input/brazilian-ecommerce/olist_order_reviews_dataset.csv')

df_pb['review_comment_title'] = df_pb['review_comment_title'].fillna('')
df_pb['review_comment_message'] = df_pb['review_comment_message'].fillna('')
df_pb['full_text'] = df_pb['review_comment_title'] + ". " + df_pb['review_comment_message']
df_pred = df_pb[df_pb['full_text'].str.len() > 3].copy()

print(f"Original Data {len(df_pb)} ，Comments containing >=3 words: {len(df_pred)} ")
print("Preparing to start AI batch prediction...")

classifier = pipeline(
    "sentiment-analysis", 
    model="./Olist_bert_final_model", 
    tokenizer="./Olist_bert_final_model", 
    device=0, 
    batch_size=64
)

texts = df_pred['full_text'].tolist()
predictions = []
for result in tqdm(classifier(texts), total=len(texts), desc="AI is reading comments"):
    predictions.append(result)

labels = [p['label'] for p in predictions]
scores = [p['score'] for p in predictions]

df_pred['sentiment_label'] = labels
df_pred['sentiment_confidence'] = scores
df_pred['sentiment_type'] = df_pred['sentiment_label'].apply(
    lambda x: 'Positive' if x == 'LABEL_1' else 'Negative'
)

output_filename = "olist_reviews_with_ai_sentiment.csv"

final_columns = [
    'review_id', 'order_id', 'review_score', 
    'review_comment_title', 'review_comment_message', 
    'sentiment_type', 'sentiment_confidence'
]

df_pred[final_columns].to_csv(output_filename, index=False)

print(f"done！")
print(f"The file has been saved as: {output_filename}")
print(f"Include: {final_columns}")

Device set to use cuda:0


Original Data 99224 ，Comments containing >=3 words: 42590 
Preparing to start AI batch prediction...


AI is reading comments:   0%|          | 0/42590 [00:00<?, ?it/s]

done！
The file has been saved as: olist_reviews_with_ai_sentiment.csv
Include: ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'sentiment_type', 'sentiment_confidence']
